# 09. 画像も入れてみる

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の実験 09 です。

Gemma 4 は **画像も読める** モデルです。これまでの実験では画像の部品を外して（`--language-model-only`）動かしてきました。
ここでは画像の部品も載せて、**グラフ・レシート・画面のスクリーンショット・図形・このリポジトリの図・写真** を読ませます。

- モデル: `cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit`（[実験04](../docs/results/04_vllm_speedup.md)・[08](../docs/results/08_rag_api.md) と同じ）
  - 言葉の部分は 4bit、画像の部品（vision tower、約 5 億個）は bf16 のまま入っている
- 動かし方: vLLM の OpenAI 互換 API サーバー（08 と同じ）。画像は ChatGPT の API と同じ `image_url` で送る
- 画像: 答えが分かっている画像をノートの中で作る（グラフ・レシート・スクショ・図形・計算）＋ このリポジトリの図（SVG）＋ 猫の写真 1 枚

測るもの:

1. 画像の質問 14 問の正答率（数字は決めた範囲に入れば正解）
2. 画像の部品を載せると、VRAM と会話の記憶（KV キャッシュ）がどれだけ減るか（08 の文字だけの設定と比べる）
3. 画像 1 枚が何トークンになるか、1 枚あたりの時間、同時に聞いたときの速さ
4. 画像の細かさ（1 枚あたりのトークン数 70 / 280 / 1120）を変えると、小さい文字の読み取りが変わるか

「想定どおり」とは:

- 画像の部品を入れても L4 に載る（サーバーが起動する）
- 14 問のうち 80% 以上正解
- 日本語の文字（レシート）を読める
- 画像 1 枚あたり 5 秒以内で答える
- 画像を細かくすると、小さい文字の問題の正解が増える（または減らない）

所要時間の目安: 20〜30 分。

---

## 実行する前に

1. **ランタイム → ランタイムのタイプを変更 → L4 GPU → Save**
2. 上から順に ▶（または「すべてのセルを実行」）
3. 終わったら **ランタイム → セッションを管理 → 解放**

## 1. GPU を確認して、ライブラリを入れる

In [ ]:
# ノート本体では GPU を使わない（GPU は vLLM の API サーバーが使う）
import subprocess
q = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True)
gpu_name, mem_mib = [x.strip() for x in q.strip().split(",")]
vram_total_gb = int(mem_mib) / 1024
assert "L4" in gpu_name, f"GPU が L4 ではありません: {gpu_name}"
print("GPU:", gpu_name, round(vram_total_gb, 1), "GB")

In [ ]:
# vLLM は専用の仮想環境に入れる（実験04・08 と同じ）
!pip install -q uv
!uv venv -q --allow-existing /content/vllm-env
!uv pip install --python /content/vllm-env/bin/python vllm 2>&1 | tail -2
!pip install -q -U openai 2>&1 | tail -2
# 画像に日本語を書くためのフォントと、SVG を PNG にする道具
!apt-get -qq install -y fonts-noto-cjk librsvg2-bin > /dev/null 2>&1
!/content/vllm-env/bin/python -c "import vllm; print('vllm', vllm.__version__)"
!fc-list | grep -c "Noto Sans CJK"

## 2. 画像の部品も載せて、API サーバーを起動する

08 との違いは 2 つです。

- **`--language-model-only` を外した**（画像の部品も載せる）
- `--max-num-batched-tokens` を 2048 → **4096**（画像 1 枚は最大 2,496 トークンになり、2048 だと `Chunked MM input disabled but max_tokens_per_mm_item (2496) is larger than max_num_batched_tokens` で起動が止まる）

- `--limit-mm-per-prompt`: 1 回の質問で送れる画像は 1 枚まで（準備するメモリを減らす）

In [ ]:
import os, re, time, requests

MODEL = "cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit"
server_log = open("/content/vllm_server.log", "w")
t0 = time.time()
server = subprocess.Popen(
    ["/content/vllm-env/bin/vllm", "serve", MODEL,
     "--served-model-name", "gemma4-26b",
     "--limit-mm-per-prompt", '{"image": 1}',
     "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.92",
     "--kv-cache-dtype", "fp8",
     "--max-num-batched-tokens", "4096",  # 08 は 2048。画像 1 枚は最大 2,496 トークンなので足りない
     "--max-num-seqs", "16",
     "--port", "8000"],
    stdout=server_log, stderr=subprocess.STDOUT,
    # 仮想環境の bin を PATH に入れる（入れないと起動の途中で FileNotFoundError: 'ninja'）
    env={**os.environ, "PATH": "/content/vllm-env/bin:" + os.environ["PATH"]})

while True:
    if server.poll() is not None:
        causes = [l for l in open("/content/vllm_server.log").read().splitlines()
                  if re.search(r"(Error|error:|out of memory)", l) and "File " not in l]
        print("\n".join(causes[-8:]))
        raise RuntimeError("vLLM サーバーが止まりました（上の原因の行、または /content/vllm_server.log を確認）")
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(5)
server_start_min = (time.time() - t0) / 60
vram_used_gb = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)) / 1024
log = open("/content/vllm_server.log").read()
kv = re.findall(r"GPU KV cache size: ([\d,]+) tokens", log)
load = re.findall(r"Model loading took ([\d.]+) ?GiB", log)
print(f"サーバー起動: {server_start_min:.1f} 分 / nvidia-smi の使用量: {vram_used_gb:.2f} GB")
print("モデルの重み:", load[-1] if load else "-", "GiB / KV キャッシュ:", kv[-1] if kv else "-", "トークン")
!grep -E "GPU KV cache size|Model loading took|encoder cache|mm_limits|profil" /content/vllm_server.log | tail -5

## 3. 答えが分かっている画像を作る

どれも **正解をノートの中で決めてから** 画像にします。

| 画像 | 中身 |
|---|---|
| 棒グラフ | 果物 5 種類の数（数字は書かない。目盛りから読む） |
| レシート | 日本語の品名と値段、合計 |
| スクリーンショット | 黒い画面に `nvidia-smi` 風の表示 |
| 図形 | 赤い丸と青い四角を散らす（数える） |
| 計算 | 大きな字で書いた掛け算 |
| リポジトリの図 | [07 の図](../docs/images/07_l4_limit.svg)・[08 の図](../docs/images/08_rag_api.svg) を PNG にしたもの |
| 写真 | Wikimedia Commons の猫の写真 |
| 小さい文字の表 | 商品コード 40 行を小さい字で（細かさの比較に使う） |

In [ ]:
import random, io, base64, urllib.request
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont

JP = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
MONO = "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf"
font_manager.fontManager.addfont(JP)
matplotlib.rcParams["font.family"] = font_manager.FontProperties(fname=JP).get_name()
os.makedirs("/content/img", exist_ok=True)
IMGS = {}

# 棒グラフ（数字は書かない）
fruits = {"りんご": 42, "みかん": 67, "ぶどう": 25, "もも": 51, "なし": 38}
fig, ax = plt.subplots(figsize=(6, 4), dpi=100)
ax.bar(list(fruits), list(fruits.values()), color="#4a90d9")
ax.set_ylim(0, 80); ax.set_yticks(range(0, 81, 10)); ax.grid(axis="y", alpha=0.4)
ax.set_title("果物の売れた数（個）")
fig.tight_layout(); fig.savefig("/content/img/bar.png"); plt.close(fig)
IMGS["bar"] = "/content/img/bar.png"

# レシート
im = Image.new("RGB", (420, 420), "white"); d = ImageDraw.Draw(im)
f = ImageFont.truetype(JP, 22); fb = ImageFont.truetype(JP, 26)
d.text((120, 20), "カフェ ひまわり", font=fb, fill="black")
d.text((40, 70), "2026年9月25日 12:34", font=f, fill="black")
items = [("ホットコーヒー", 450), ("たまごサンド", 680), ("季節のサラダ", 520)]
y = 130
for name, p in items:
    d.text((40, y), name, font=f, fill="black"); d.text((300, y), f"¥{p:,}", font=f, fill="black"); y += 45
d.line((30, y + 5, 390, y + 5), fill="black", width=2)
d.text((40, y + 20), "合計", font=fb, fill="black"); d.text((290, y + 20), f"¥{sum(p for _, p in items):,}", font=fb, fill="black")
im.save("/content/img/receipt.png"); IMGS["receipt"] = "/content/img/receipt.png"

# スクリーンショット（nvidia-smi 風）
lines = ["$ nvidia-smi",
         "+-----------------------------------------------------------+",
         "| GPU  Name        Temp  Power     Memory-Usage       Util  |",
         "|   0  NVIDIA L4   61C   58W/72W   17034MiB / 23034MiB  87% |",
         "+-----------------------------------------------------------+",
         "| Processes:  PID 4121  python3 (vllm)      16912MiB        |",
         "+-----------------------------------------------------------+"]
fm = ImageFont.truetype(MONO, 16)
im = Image.new("RGB", (640, 40 + 24 * len(lines)), (30, 30, 30)); d = ImageDraw.Draw(im)
for i, l in enumerate(lines):
    d.text((16, 20 + 24 * i), l, font=fm, fill=(220, 220, 220))
im.save("/content/img/terminal.png"); IMGS["terminal"] = "/content/img/terminal.png"

# 図形（重ならないように置く）
rng = random.Random(9)
im = Image.new("RGB", (600, 400), "white"); d = ImageDraw.Draw(im)
placed = []
def place():
    while True:
        x, y = rng.randint(40, 560), rng.randint(40, 360)
        if all((x - a) ** 2 + (y - b) ** 2 > 90 ** 2 for a, b in placed):
            placed.append((x, y)); return x, y
for _ in range(7):
    x, y = place(); d.ellipse((x - 28, y - 28, x + 28, y + 28), fill=(220, 40, 40))
for _ in range(4):
    x, y = place(); d.rectangle((x - 26, y - 26, x + 26, y + 26), fill=(40, 80, 220))
im.save("/content/img/shapes.png"); IMGS["shapes"] = "/content/img/shapes.png"

# 計算
im = Image.new("RGB", (500, 200), "white"); d = ImageDraw.Draw(im)
d.text((60, 50), "23 × 17 = ?", font=ImageFont.truetype(JP, 72), fill="black")
im.save("/content/img/math.png"); IMGS["math"] = "/content/img/math.png"

# このリポジトリの図（SVG → PNG）
RAW = "https://raw.githubusercontent.com/moruku36/colab-oss-lab/main/"
for key, path in [("svg07", "docs/images/07_l4_limit.svg"), ("svg08", "docs/images/08_rag_api.svg")]:
    open(f"/content/img/{key}.svg", "wb").write(urllib.request.urlopen(RAW + path).read())
    subprocess.run(["rsvg-convert", "-w", "1290", "-b", "white", f"/content/img/{key}.svg", "-o", f"/content/img/{key}.png"], check=True)
    IMGS[key] = f"/content/img/{key}.png"

# 写真（Wikimedia Commons の猫）
req = urllib.request.Request("https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg",
                             headers={"User-Agent": "colab-oss-lab/0.1 (https://github.com/moruku36/colab-oss-lab)"})
Image.open(io.BytesIO(urllib.request.urlopen(req).read())).convert("RGB").save("/content/img/photo.jpg")
IMGS["photo"] = "/content/img/photo.jpg"

# 小さい文字の表（40 行、大きい画像に小さい字）
rng2 = random.Random(2026)
codes = [(f"{rng2.choice('ABCDEFGH')}-{rng2.randint(1000, 9999)}", rng2.randint(100, 9999)) for _ in range(40)]
fs = ImageFont.truetype(JP, 13)
im = Image.new("RGB", (1400, 1400), "white"); d = ImageDraw.Draw(im)
d.text((40, 20), "商品コードと価格（円）", font=ImageFont.truetype(JP, 18), fill="black")
for i, (c, p) in enumerate(codes):
    col, row = divmod(i, 20)
    d.text((40 + col * 700, 70 + row * 32), f"{i + 1:02d}  {c}   {p:,} 円", font=fs, fill="black")
im.save("/content/img/dense.png"); IMGS["dense"] = "/content/img/dense.png"
dense_qs = [codes[5], codes[17], codes[26], codes[38]]

for k, p in IMGS.items():
    print(k, Image.open(p).size)

In [ ]:
# 作った画像を並べて見る
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, (k, p) in zip(axes.flat, IMGS.items()):
    ax.imshow(Image.open(p)); ax.set_title(k); ax.axis("off")
for ax in axes.flat[len(IMGS):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. 画像を API で送る

ChatGPT の API と同じく、画像は `{"type": "image_url", "image_url": {"url": "data:image/png;base64,..."}}` で送ります。

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")  # 鍵は使わない（自分の Colab の中だけ）

def data_url(path):
    mime = "image/jpeg" if path.endswith(".jpg") else "image/png"
    return f"data:{mime};base64," + base64.b64encode(open(path, "rb").read()).decode()

def ask_image(key, q, max_tokens=200, soft_tokens=None):
    extra = {"chat_template_kwargs": {"enable_thinking": False}}
    if soft_tokens:
        extra["mm_processor_kwargs"] = {"max_soft_tokens": soft_tokens}
    msgs = [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": data_url(IMGS[key])}},
        {"type": "text", "text": q}]}]
    t = time.time()
    r = client.chat.completions.create(model="gemma4-26b", messages=msgs, max_tokens=max_tokens,
                                       temperature=0, extra_body=extra)
    return dict(answer=r.choices[0].message.content.strip(), sec=time.time() - t,
                in_tokens=r.usage.prompt_tokens, out_tokens=r.usage.completion_tokens)

# 1 枚の画像が何トークンになるか: 同じ質問を文字だけで送ったときと比べる
r_txt = client.chat.completions.create(model="gemma4-26b", max_tokens=1, temperature=0,
                                       messages=[{"role": "user", "content": [{"type": "text", "text": "この画像を説明してください。"}]}],
                                       extra_body={"chat_template_kwargs": {"enable_thinking": False}})
first = ask_image("svg07", "この画像を説明してください。", max_tokens=300)
image_tokens = first["in_tokens"] - r_txt.usage.prompt_tokens
print(f"画像 1 枚 ≒ {image_tokens} トークン / {first['sec']:.1f} 秒")
print(first["answer"])

## 5. 14 問で評価する

In [ ]:
def nums(text):
    return [float(x) for x in re.findall(r"\d+(?:\.\d+)?", text.replace(",", "").replace("，", ""))]

def judge(ans, spec):
    kind, val = spec
    if kind == "kw":      # どれかの言葉が入っていれば正解
        return any(v in ans for v in val)
    lo, hi = val          # 数字が範囲に入っていれば正解
    return any(lo <= x <= hi for x in nums(ans))

QA = [
    ("bar", "グラフでいちばん多い果物はどれですか。名前だけ答えてください。", ("kw", ["みかん"])),
    ("bar", "グラフで、ぶどうはおよそ何個ですか。数字だけ答えてください。", ("num", (22, 28))),
    ("bar", "グラフで、ももはおよそ何個ですか。数字だけ答えてください。", ("num", (48, 54))),
    ("receipt", "たまごサンドはいくらですか。", ("num", (680, 680))),
    ("receipt", "合計金額はいくらですか。", ("num", (1650, 1650))),
    ("receipt", "このお店の名前は何ですか。", ("kw", ["ひまわり"])),
    ("terminal", "この画面の GPU の名前は何ですか。", ("kw", ["L4"])),
    ("terminal", "GPU のメモリは何 MiB 使われていますか。数字だけ答えてください。", ("num", (17034, 17034))),
    ("shapes", "赤い丸はいくつありますか。数字だけ答えてください。", ("num", (7, 7))),
    ("shapes", "青い四角はいくつありますか。数字だけ答えてください。", ("num", (4, 4))),
    ("math", "画像の計算の答えはいくつですか。数字だけ答えてください。", ("num", (391, 391))),
    ("svg07", "この図で、Seed-OSS-36B の VRAM は何 GB ですか。", ("num", (19.64, 19.64))),
    ("svg08", "この図で、RAG ありの正答率は何 % ですか。", ("num", (64, 64))),
    ("photo", "この写真に写っている動物は何ですか。一言で答えてください。", ("kw", ["猫", "ネコ", "ねこ", "cat", "Cat"])),
]
rows = []
for key, q, spec in QA:
    a = ask_image(key, q, max_tokens=100)
    rows.append(dict(img=key, q=q, answer=a["answer"], ok=judge(a["answer"], spec), sec=a["sec"],
                     in_tokens=a["in_tokens"], out_tokens=a["out_tokens"]))
    print("○" if rows[-1]["ok"] else "×", key, "|", q, "→", a["answer"][:60].replace("\n", " "), f"({a['sec']:.1f}s)")
acc = sum(r["ok"] for r in rows) / len(rows)
receipt_ok = all(r["ok"] for r in rows if r["img"] == "receipt")
avg_sec = sum(r["sec"] for r in rows) / len(rows)
print(f"正答率 {acc:.0%} / 1 問あたり {avg_sec:.2f} 秒")

In [ ]:
# 説明させる（採点しない。返事の例として残す）
desc = {k: ask_image(k, q, max_tokens=300) for k, q in [
    ("photo", "この写真を日本語で 2 文で説明してください。"),
    ("terminal", "この画面から分かることを、初心者向けに 3 文で説明してください。"),
    ("svg08", "この図が伝えたいことを 3 文でまとめてください。"),
]}
for k, v in desc.items():
    print(f"--- {k} ({v['sec']:.1f}s)\n{v['answer']}\n")

## 6. 同時に聞いたときの速さ

14 問を 1 件ずつ順番に聞く場合と、同時に聞く場合を比べます。

In [ ]:
from concurrent.futures import ThreadPoolExecutor
t = time.time()
for key, q, _ in QA:
    ask_image(key, q, max_tokens=100)
seq_sec = time.time() - t
t = time.time()
with ThreadPoolExecutor(max_workers=14) as ex:
    par = list(ex.map(lambda x: ask_image(x[0], x[1], max_tokens=100), QA))
par_sec = time.time() - t
print(f"順番に 14 問: {seq_sec:.1f} 秒 / 同時に 14 問: {par_sec:.1f} 秒（{seq_sec / par_sec:.1f} 倍）")

## 7. 画像の細かさを変える

Gemma 4 は、画像 1 枚を何トークンに縮めるか（`max_soft_tokens`）を 70 / 140 / 280 / 560 / 1120 から選べます。
小さい文字の表（1400×1400 に 13px の字で 40 行）で、細かさを変えて読み取りを比べます。

In [ ]:
res_rows = []
for st in [70, 280, 1120]:
    try:
        got = []
        for c, p in dense_qs:
            a = ask_image("dense", f"商品コード {c} の価格は何円ですか。数字だけ答えてください。", max_tokens=30, soft_tokens=st)
            got.append((c, p, a))
        n_ok = sum(judge(a["answer"], ("num", (p, p))) for c, p, a in got)
        rc = ask_image("receipt", "合計金額はいくらですか。", max_tokens=30, soft_tokens=st)
        res_rows.append(dict(soft=st, tokens=got[0][2]["in_tokens"], dense_ok=n_ok, sec=sum(a["sec"] for *_, a in got) / len(got),
                             receipt_ok=judge(rc["answer"], ("num", (1650, 1650))),
                             answers=[f"{c}: 正解 {p} → {a['answer']}" for c, p, a in got]))
        print(st, res_rows[-1])
    except Exception as e:
        print(st, "この設定は使えませんでした:", str(e)[:200])
        res_rows.append(dict(soft=st, error=str(e)[:200]))
res_ok = [r for r in res_rows if "error" not in r]
res_changed = len({r["tokens"] for r in res_ok}) > 1

## 8. まとめて、実行記録を出す

In [ ]:
from datetime import datetime, timezone, timedelta
vv = subprocess.check_output(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"], text=True).strip()
res_check = (res_changed and res_ok[-1]["dense_ok"] >= res_ok[0]["dense_ok"])
checks = {
    "画像の部品を入れても L4 に載る（サーバーが起動する）": True,
    "14 問のうち 80% 以上正解": acc >= 0.80,
    "日本語の文字（レシート）を読める": receipt_ok,
    "画像 1 枚あたり 5 秒以内で答える": avg_sec <= 5,
    "画像を細かくすると、小さい文字の問題の正解が増える（または減らない）": res_check,
}
ok = all(checks.values())
now = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")
L = ["# 実行記録: 09 画像も入れてみる", "",
     f"- 実行日: {now}", "- 実行場所: Google Colab",
     f"- GPU: {gpu_name} / VRAM {round(vram_total_gb, 1)} GB",
     f"- モデル: {MODEL}（vLLM {vv} の OpenAI 互換 API サーバー、画像の部品あり、thinking オフ、temperature 0）",
     "- 08 からの変更: `--language-model-only` を外す / `--max-num-batched-tokens` 2048 → 4096",
     f"- サーバー起動: {server_start_min:.1f} 分 / 重み: {load[-1] if load else '-'} GiB / KV キャッシュ: {kv[-1] if kv else '-'} トークン（08 の文字だけ: 59,412）/ nvidia-smi: {vram_used_gb:.2f} GB",
     f"- 画像 1 枚 ≒ {image_tokens} トークン（初期設定）",
     f"- 想定どおりか: {'はい' if ok else 'いいえ'}", "",
     "## まとめ", "",
     f"- 正答率: **{acc:.0%}**（{sum(r['ok'] for r in rows)} / {len(rows)}）",
     f"- 1 問あたり: {avg_sec:.2f} 秒 / 入力 {sum(r['in_tokens'] for r in rows) / len(rows):.0f} トークン",
     f"- 14 問を順番に: {seq_sec:.1f} 秒 / 同時に: {par_sec:.1f} 秒（{seq_sec / par_sec:.1f} 倍速い）", "",
     "## 判定", ""] + [f"- [{'x' if v else ' '}] {k}" for k, v in checks.items()] + ["",
     "## 問題ごと", "", "| 画像 | 質問 | 正誤 | 答え | 秒 |", "|---|---|---|---|---|"]
for r in rows:
    L.append(f"| {r['img']} | {r['q']} | {'○' if r['ok'] else '×'} | {r['answer'][:80].replace(chr(10), ' ')} | {r['sec']:.1f} |")
L += ["", "## 画像の細かさ（小さい文字の表 4 問 + レシートの合計）", "", "| max_soft_tokens | 入力トークン | 小さい文字 | レシート | 秒 |", "|---|---|---|---|---|"]
for r in res_rows:
    if "error" in r:
        L.append(f"| {r['soft']} | 使えなかった: {r['error'][:80]} | | | |")
    else:
        L.append(f"| {r['soft']} | {r['tokens']} | {r['dense_ok']} / 4 | {'○' if r['receipt_ok'] else '×'} | {r['sec']:.1f} |")
for r in res_ok:
    L += ["", f"{r['soft']}: " + " / ".join(a.replace(chr(10), ' ')[:60] for a in r["answers"])]
L += ["", "## 説明させた例", ""]
for k, v in [("svg07", first)] + list(desc.items()):
    L += [f"### {k}", "", "```", v["answer"][:600], "```", ""]
print("\n".join(L))

## 9. サーバーを止める

終わったら vLLM のサーバーを止めて、**ランタイム → セッションを管理 → 解放** を押してください。

In [ ]:
server.terminate()
server.wait(timeout=60)
print("サーバーを止めました")